In [1]:
import cv2

In [2]:
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TerminateOnNaN, CSVLogger
from keras import backend as K
from keras.models import load_model
from math import ceil
import numpy as np
from matplotlib import pyplot as plt, patches
from sklearn.model_selection import train_test_split

from models.ssd7_custom import build_model
from models.ssd7_resnet_backbone import resnet_build_model
from models.ssd300_custom import ssd300_build_model
from loss_function.custom_loss import AOILoss
from custom_layers.GridCenters import GridCenters

from input_encoder_decoder.input_encoder import SSDInputEncoder
from input_encoder_decoder.output_decoder import decode_detections
from input_encoder_decoder.data_generator import DataGenerator

%matplotlib inline

2023-01-05 14:38:58.661944: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-05 14:38:59.130485: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/token/miniconda3/envs/tf/lib/python3.10/site-packages/cv2/../../lib64::/home/token/miniconda3/envs/tf/lib/:/home/token/miniconda3/envs/tf/lib/:/home/token/miniconda3/envs/tf/lib/
2023-01-05 14:38:59.130528: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared ob

In [3]:
model = load_model("model_realistic_pcb_resnetv2_conv1", custom_objects={'GridCenters': GridCenters,
                                                           'compute_loss': AOILoss})


2023-01-05 14:39:00.137060: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-05 14:39:00.140464: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-05 14:39:00.140592: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-05 14:39:00.140882: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorF

In [4]:
img_width = 300
img_height= 300

In [5]:
def plot_detections(decoded_pred, image):
    if decoded_pred.shape[0] == 0:
        return image
    
    #plt.figure(figsize=(10,6))
    #plt.imshow(image)
    #current_axis = plt.gca()

    #colors = plt.cm.hsv(np.linspace(0, 1, n_classes+1)).tolist() # Set the colors for the bounding boxes
    classes = ['background', 'ic'] # Just so we can print class names onto the image instead of IDs

    for label in decoded_pred:
        pred = np.array(label)
        corners = np.reshape(pred[2:], (-1, 2)).astype(int)
        #color = colors[int(pred[0])]
        label_text = '{}: {:.2f}'.format(classes[int(pred[0])], pred[1])
        for points in corners:
            image = cv2.circle(image, tuple(points), 2, (0,0,255), 1)
            #current_axis.add_patch(plt.Circle(tuple(points), 2, fill=False, color='red'))
        center_x = (corners[0, 0] + corners[3, 0]) / 2
        center_y = (corners[0, 1] + corners[3, 1]) / 2
        #current_axis.text(center_x, center_y, label, size='x-small', color='white', bbox={'facecolor':color, 'alpha':1.0})
        font = cv2.FONT_HERSHEY_SIMPLEX

        # fontScale
        fontScale = 1

        # Blue color in BGR
        color = (255, 0, 0)

        # Line thickness of 2 px
        thickness = 2
        
        image = cv2.putText(image, label_text, (int(center_x),int(center_y)), font, fontScale, color, thickness, cv2.LINE_AA)
        
    return image   

In [6]:
cam = cv2.VideoCapture(0)
while True:
    ret_val, img = cam.read()
    if ret_val:
        #print(img.shape) (480, 640, 3)
        img = img[0:300, 0:300]
        img = cv2.resize(img, (img_height,img_width), interpolation = cv2.INTER_AREA)
        img_pred = img.copy()
        img_pred = cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB)
        predictions = model.predict([np.expand_dims(img_pred, axis=0)], verbose=0)
        #print(predictions)
        decoded_pred = decode_detections(predictions, img_height=img_height, img_width=img_width)
        #print(decoded_pred[0])
        img = plot_detections(decoded_pred[0],img)
        #break
        img = cv2.resize(img, (img_height*2,img_width*2), interpolation = cv2.INTER_AREA)
        cv2.imshow('AI AOI', img)
    if cv2.waitKey(1) == 27: 
        break  # esc to quit
cv2.destroyAllWindows()

2023-01-05 14:39:06.743755: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8100
2023-01-05 14:39:07.176007: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2023-01-05 14:39:07.176439: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2023-01-05 14:39:07.176467: W tensorflow/compiler/xla/stream_executor/gpu/asm_compiler.cc:85] Couldn't get ptxas version string: INTERNAL: Couldn't invoke ptxas --version
2023-01-05 14:39:07.176986: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2023-01-05 14:39:07.177067: W tensorflow/compiler/xla/stream_executor/gpu/redzone_allocator.cc:318] INTERNAL: Failed to launch ptxas
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be only logged once.
QObject::moveToThread: Curr